# Pilot 4 · rank-1 GRPO arm

**Current implementation: source preparation and audit only (ticket 01).** Training, baseline sampling, GPU preflight, evaluation and measurement will be added by the remaining tickets. This notebook does not yet launch the scientific run.

Pilot 2 trained rank-1 SFT on 512 GSM8K problems. Pilot 4 will use those same problems and optimizer prompt order, changing to group-normalized reward training. We first verify the saved evidence so missing or changed cohorts cannot silently change the experiment.

**No GPU is needed for these cells.** The sources stay in Drive; no model or dataset is downloaded during preparation. FP32 for both arms is approved. Historical accuracy 0.640 is contextual, not a correctness gate.

## 1. Mount Drive and select the checked-out implementation

The code revision must be available on GitHub before Colab can fetch it. Use a separate idle runtime; do not switch a checkout while another experiment subprocess is using it. Local edits are preserved by refusing checkout until you inspect them.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
HARNESS_COMMIT = '55fe8a1c0f2a917e3ee4ad4855db35f999c5555e'
REPO_DIR = Path('/content/spar-pilot')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts')
SFT_PLAN = 'pilot2-sft-fp32-l4-batch8-v1' # @param {type:"string"}
MEASUREMENT_PLAN = 'pilot3-inputs-v1' # @param {type:"string"}
PLAN_NAME = 'pilot4-grpo-fp32-v1' # @param {type:"string"}
SFT_CONFIG = ARTIFACT_ROOT / 'plans' / SFT_PLAN / 'sft.config.json'
MEASUREMENT_CONFIG = ARTIFACT_ROOT / 'plans' / MEASUREMENT_PLAN / 'activation.prepared.json'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert not dirty, 'Preserve local repository edits before changing the checkout.'
subprocess.run(['git', 'fetch', 'origin', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
for source_path in [SFT_CONFIG, MEASUREMENT_CONFIG]:
    print(source_path, '| exists:', source_path.is_file())
    assert source_path.is_file(), 'Locate the saved source config in Drive before proceeding.'


## 2. Verify sources and prepare the plan

This calls the same public command tested offline. It verifies completed SFT checkpoints, frozen-base evidence, original training/evaluation data, actual saved prompt order and Pilot 3 token/mask hashes. It reuses those files rather than resampling or tokenizing again.

**Expected:** a prepared config, matching audit, metadata and completion marker. Repeating this cell revalidates/reuses the preparation. An incompatible source or corrupt artifact stops the command. Prepared source evidence does not mean training has run or the scientific protocol is frozen.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-prepare',
                '--sft-config', str(SFT_CONFIG), '--measurement-config', str(MEASUREMENT_CONFIG),
                '--name', PLAN_NAME, '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
PREPARED_PATH = ARTIFACT_ROOT / 'plans' / PLAN_NAME / 'grpo.prepared.json'


## 3. Read the matching audit

Review the identities and intended matches: 512 training problems, 150 held-out items, 150 GSM8K + 150 FineWeb measurement sequences, FP32 and rank-1 adapters. The saved SFT optimizer order is reused; the selected source indices alone do not specify training order.

The unresolved table is intentional. It identifies the scorer confirmation, exact verified loss, baseline subset/generation policy, final completion cap, monitor actions and preregistration. Later tickets resolve them before a full run. Historical aggregate results are not tests of this preparation.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-audit',
                '--config', str(PREPARED_PATH), '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
PLAN = json.loads(PREPARED_PATH.read_text())
print('Prepared config:', PREPARED_PATH)
print('Audit:', ARTIFACT_ROOT / PLAN['audit_path'])
print('Pending choices:', ', '.join(PLAN['pending']))
print('Next stage: untuned baseline sampling after its settings are explicitly resolved.')


## What follows

The remaining implementation adds tiny-model controls, baseline sampling, real GPU preflight and preregistration freeze, 64-step training, greedy/sampled evaluation, fixed-token write measurements and the SFT–GRPO comparison.

Training will resume only from sealed checkpoints 0, 8, 16, 32 and 64. The final cap is chosen from uncensored baseline evidence. Until those stages exist and all gates pass, keep this plan as prepared inputs, not a completed scientific experiment.
